# Sequential Penalty Transformation Methods: Barrier and Penalty

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed-Bayoumy/MECH559/blob/DEV/L09/L09_penalty_and_barrier_methods.ipynb)
### MECH 559 - Systems Optimization - L09

Companion notebook to the "Algorithms for nonlinearly constrained optimization" lecture - this fills in the numerical example the slides point to ("Comparing the penalty and barrier function methods"). Both methods turn a constrained problem into a *sequence* of unconstrained problems: the barrier method stays strictly feasible and blows up at the boundary, while the penalty method allows infeasibility and only punishes it.

## Learning objectives

By the end, students should be able to:

1. Implement the interior barrier function $T(x,r)=f(x)+rB(x)$ and the exterior quadratic penalty $T(x,r)=f(x)+\tfrac1r P(x)$, both driven by $r_k\to0$.
2. Explain why the barrier method needs a strictly feasible starting point and the penalty method does not.
3. Observe numerically how the sequence of unconstrained minimizers approaches the true constrained optimum as $r_k\to0$, and why very small $r$ causes ill-conditioning.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

plt.rcParams.update({
    "font.size": 11,
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

COLORS = {"barrier": "#1D5FA8", "penalty": "#D1495B", "optimum": "#2A9D5C", "feasible": "#A8DADC"}
print(f"NumPy {np.__version__}, SciPy ok")

NumPy 2.5.0, SciPy ok


---
## 1. The example problem (from the slides)

$$
\min_{x_1,x_2}\; f(x_1,x_2) = 3x_1^2+x_2^2
\qquad\text{s.t.}\qquad
g_1(x_1,x_2) = 2-x_2-x_1 \le 0.
$$

The unconstrained minimizer $(0,0)$ has $g_1(0,0)=2>0$: infeasible, so the constraint is active at the optimum. On the boundary $x_1+x_2=2$, minimizing $3x_1^2+(2-x_1)^2$ gives $x_1^\star=0.5,\ x_2^\star=1.5$, $f^\star=3.0$ - our analytic reference.

In [2]:
def f(x):
    return 3.0 * x[0] ** 2 + x[1] ** 2

def grad_f(x):
    return np.array([6.0 * x[0], 2.0 * x[1]])

def g1(x):
    return 2.0 - x[1] - x[0]

x_star = np.array([0.5, 1.5])
print(f"analytic optimum x* = {x_star},  f(x*) = {f(x_star):.4f},  g1(x*) = {g1(x_star):.2e}")

analytic optimum x* = [0.5 1.5],  f(x*) = 3.0000,  g1(x*) = 0.00e+00


---
## 2. Barrier method

$$
T(x,r) = f(x) + r\,B(x), \qquad B(x) = -\frac{1}{g_1(x)}, \qquad r_k \to 0.
$$

$B(x)>0$ inside the feasible region and $B(x)\to+\infty$ as $x\to\partial\{g_1=0\}$. We warm-start each unconstrained minimization from the previous $r_k$'s solution, beginning strictly inside the feasible region.

In [3]:
def T_barrier(x, r):
    if g1(x) >= 0:
        return np.inf
    return f(x) + r * (-1.0 / g1(x))

def run_barrier(x0, r0=2.0, beta=0.3, n_outer=14):
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    for k in range(n_outer):
        r = r0 * beta ** k
        res = minimize(T_barrier, x, args=(r,), method="Nelder-Mead",
                        options={"xatol": 1e-10, "fatol": 1e-12, "maxiter": 5000})
        x = res.x
        path.append(x.copy())
    return np.array(path)

barrier_path = run_barrier(x0=[2.0, 2.0])   # g1(2,2) = -2 < 0: strictly feasible start
print("barrier sequence (every other iterate):")
for k, x in list(enumerate(barrier_path))[::2]:
    print(f"  k={k:2d}  x=({x[0]:.5f}, {x[1]:.5f})  f={f(x):.5f}  g1={g1(x): .2e}")
print(f"\nfinal: x = {barrier_path[-1]}, error to analytic optimum = {np.linalg.norm(barrier_path[-1]-x_star):.2e}")

barrier sequence (every other iterate):
  k= 0  x=(2.00000, 2.00000)  f=16.00000  g1=-2.00e+00
  k= 2  x=(0.60190, 1.80570)  f=4.34741  g1=-4.08e-01
  k= 4  x=(0.53250, 1.59750)  f=3.40269  g1=-1.30e-01
  k= 6  x=(0.50996, 1.52989)  f=3.12075  g1=-3.99e-02
  k= 8  x=(0.50301, 1.50903)  f=3.03622  g1=-1.20e-02
  k=10  x=(0.50090, 1.50271)  f=3.01087  g1=-3.62e-03
  k=12  x=(0.50027, 1.50081)  f=3.00326  g1=-1.09e-03
  k=14  x=(0.50008, 1.50024)  f=3.00098  g1=-3.26e-04

final: x = [0.50008151 1.50024449], error to analytic optimum = 2.58e-04


---
## 3. Penalty method

$$
T(x,r) = f(x) + \frac1r P(x), \qquad P(x)=\max\{0,g_1(x)\}^2, \qquad r_k \to 0.
$$

$P(x)=0$ everywhere feasible and grows only outside; unlike the barrier, the penalty method tolerates - and here, starts from - an **infeasible** point.

In [4]:
def T_penalty(x, r):
    return f(x) + (1.0 / r) * max(0.0, g1(x)) ** 2

def run_penalty(x0, r0=2.0, beta=0.3, n_outer=14):
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    for k in range(n_outer):
        r = r0 * beta ** k
        res = minimize(T_penalty, x, args=(r,), method="BFGS")
        x = res.x
        path.append(x.copy())
    return np.array(path)

penalty_path = run_penalty(x0=[0.0, 0.0])   # g1(0,0) = 2 > 0: infeasible start
print("penalty sequence (every other iterate):")
for k, x in list(enumerate(penalty_path))[::2]:
    print(f"  k={k:2d}  x=({x[0]:.5f}, {x[1]:.5f})  f={f(x):.5f}  g1={g1(x): .2e}")
print(f"\nfinal: x = {penalty_path[-1]}, error to analytic optimum = {np.linalg.norm(penalty_path[-1]-x_star):.2e}")

penalty sequence (every other iterate):
  k= 0  x=(0.00000, 0.00000)  f=0.00000  g1= 2.00e+00
  k= 2  x=(0.34483, 1.03448)  f=1.42687  g1= 6.21e-01
  k= 4  x=(0.48054, 1.44161)  f=2.77100  g1= 7.78e-02
  k= 6  x=(0.49818, 1.49455)  f=2.97825  g1= 7.26e-03
  k= 8  x=(0.49984, 1.49951)  f=2.99803  g1= 6.56e-04
  k=10  x=(0.49999, 1.49996)  f=2.99982  g1= 5.91e-05
  k=12  x=(0.50000, 1.50000)  f=2.99998  g1= 5.32e-06
  k=14  x=(0.50000, 1.50000)  f=3.00000  g1= 4.86e-07

final: x = [0.49999904 1.50000048], error to analytic optimum = 1.07e-06


---
## 4. Side-by-side: two very different roads to the same point

In [5]:
x1g, x2g = np.meshgrid(np.linspace(-0.5, 2.5, 300), np.linspace(-0.5, 2.5, 300))
Fg = 3.0 * x1g ** 2 + x2g ** 2
boundary_x = np.linspace(-0.5, 2.5, 10)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, path, label, color, start_note in [
    (axes[0], barrier_path, "Barrier", COLORS["barrier"], "starts feasible, stays feasible"),
    (axes[1], penalty_path, "Penalty", COLORS["penalty"], "starts infeasible, approaches from outside"),
]:
    ax.contour(x1g, x2g, Fg, levels=25, cmap="Greys", alpha=0.5)
    ax.plot(boundary_x, 2 - boundary_x, color="black", lw=1.5, label=r"$g_1=0$")
    ax.fill_between(boundary_x, 2 - boundary_x, 2.5, color=COLORS["feasible"], alpha=0.25)
    ax.plot(path[:, 0], path[:, 1], "-o", color=color, ms=5, label=f"{label} iterates")
    ax.plot(*x_star, "*", color=COLORS["optimum"], ms=20, label="analytic optimum")
    ax.set(xlabel="$x_1$", ylabel="$x_2$", aspect="equal",
           title=f"{label} method ({start_note})")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

C:\Users\ahbay\AppData\Local\Temp\ipykernel_31248\1497633382.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
errs_b = np.linalg.norm(barrier_path - x_star, axis=1)
errs_p = np.linalg.norm(penalty_path - x_star, axis=1)

fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogy(errs_b, "-o", color=COLORS["barrier"], label="barrier")
ax.semilogy(errs_p, "-s", color=COLORS["penalty"], label="penalty")
ax.set(xlabel="outer iteration $k$ ($r_k=r_0\\beta^k$)", ylabel=r"$\|x_k-x^\star\|$",
       title="Both sequences converge as $r_k\\to0$")
ax.legend()
plt.tight_layout()
plt.show()

print(f"barrier final error : {errs_b[-1]:.2e}")
print(f"penalty final error : {errs_p[-1]:.2e}")
assert errs_b[-1] < 1e-3 and errs_p[-1] < 1e-3
print("\nBoth methods converge to the analytic constrained optimum.")

barrier final error : 2.58e-04
penalty final error : 1.07e-06

Both methods converge to the analytic constrained optimum.


C:\Users\ahbay\AppData\Local\Temp\ipykernel_31248\3342405069.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Takeaways

1. Both methods reduce the constrained problem to a *sequence* of unconstrained problems, driven by the same limit $r_k\to0$; only the way the constraint enters $T(x,r)$ differs.
2. The **barrier** method requires a strictly feasible starting point and every iterate stays feasible - useful when infeasible designs are unacceptable (e.g. cannot be evaluated), at the cost of needing an interior starting guess.
3. The **penalty** method tolerates infeasible iterates (including the starting point) and only *punishes* constraint violation - simpler to start, but the optimum is approached from outside the feasible region.
4. As $r_k\to0$, $B(x)$'s curvature near the boundary and $\tfrac1r P(x)$'s curvature both grow without bound, which is exactly the "increasing eccentricity" mentioned in the slides - this is also why very small $r$ causes numerical ill-conditioning, motivating the augmented Lagrangian method in the next notebook.